# Mounting using App

In [0]:
spark.conf.set("fs.azure.account.auth.type.<storage-account>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.<storage-account>.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<storage-account>.dfs.core.windows.net", "<application-id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<storage-account>.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<storage-account>.dfs.core.windows.net", "https://login.microsoftonline.com/<directory-id>/oauth2/token")

# Reading Bronze layer data

In [0]:
df_transaction=spark.read.format('parquet').option('header', True).option('inferSchema',True).load('abfss://retail@ecommerceadlsaccount.dfs.core.windows.net/bronze/transaction/transaction')
df_product=spark.read.format('parquet').option('header', True).option('inferSchema',True).load('abfss://retail@ecommerceadlsaccount.dfs.core.windows.net/bronze/product/product')
df_customer=spark.read.format('parquet').option('header', True).option('inferSchema',True).load('abfss://retail@ecommerceadlsaccount.dfs.core.windows.net/bronze/customer/customer')
df_store=spark.read.format('parquet').option('header', True).option('inferSchema',True).load('abfss://retail@ecommerceadlsaccount.dfs.core.windows.net/bronze/store/store')

# Create Silver Layer data - Data Cleaning

## Convert types and clean data

In [0]:
from pyspark.sql.functions import col

# Convert types and clean data
df_transaction = df_transaction.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date")
)

df_product = df_product.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

df_store = df_store.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)

df_customer = df_customer.select(
    "customer_id", "first_name", "last_name", "email", "city", "registration_date"
).dropDuplicates(["customer_id"])

In [0]:
from pyspark.sql.functions import concat, lit

df_customer=df_customer.withColumn("full_name", concat(col("first_name"), lit(" "), col("last_name"))) \
    .drop("first_name", "last_name")

In [0]:
display(df_customer)

customer_id,email,city,registration_date,full_name
101,user101@example.com,Delhi,2023-09-14,Ravi Yadav
102,user102@example.com,Mumbai,2024-01-21,Nina Joshi
103,user103@example.com,Bangalore,2023-07-10,Sonal Sharma
104,user104@example.com,Hyderabad,2024-02-05,Karan Patel
105,user105@example.com,Chennai,2023-06-28,Riya Singh
106,user106@example.com,Pune,2024-03-10,Ajay Mishra
107,user107@example.com,Ahmedabad,2023-05-12,Priya Kapoor
108,user108@example.com,Kolkata,2023-08-19,Rahul Verma
109,user109@example.com,Delhi,2024-04-01,Pooja Mehta
110,user110@example.com,Mumbai,2023-10-14,Deepak Nair


##  Join all data

In [0]:
from pyspark.sql.functions import round
df_silver = df_transaction \
    .join(df_customer, "customer_id") \
    .join(df_product, "product_id") \
    .join(df_store, "store_id") \
    .withColumn("total_amount", round(col("quantity") * col("price"), 2))

display(df_silver)

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,email,city,registration_date,full_name,product_name,category,price,store_name,location,total_amount
5,7,101,28,3,2024-11-15,user101@example.com,Delhi,2023-09-14,Ravi Yadav,Smartwatch,Electronics,4999.0,Mega Plaza,Chennai,14997.0
3,1,102,11,2,2024-08-11,user102@example.com,Mumbai,2024-01-21,Nina Joshi,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,1599.98
4,1,103,18,3,2024-09-05,user103@example.com,Bangalore,2023-07-10,Sonal Sharma,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.97
3,3,104,13,4,2025-05-04,user104@example.com,Hyderabad,2024-02-05,Karan Patel,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
3,1,105,21,5,2024-10-02,user105@example.com,Chennai,2023-06-28,Riya Singh,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,3999.95
2,5,105,5,1,2025-03-17,user105@example.com,Chennai,2023-06-28,Riya Singh,Notebook Set,Stationery,149.0,High Street Store,Delhi,149.0
4,3,105,2,5,2024-11-12,user105@example.com,Chennai,2023-06-28,Riya Singh,Yoga Mat,Fitness,499.0,Downtown Mini Store,Pune,2495.0
3,9,107,22,4,2024-11-16,user107@example.com,Ahmedabad,2023-05-12,Priya Kapoor,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
1,5,108,12,4,2025-05-26,user108@example.com,Kolkata,2023-08-19,Rahul Verma,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
5,8,109,17,5,2024-07-10,user109@example.com,Delhi,2024-04-01,Pooja Mehta,Desk Organizer,Accessories,399.0,Mega Plaza,Chennai,1995.0


## Store Cleaned data into Silver layer

In [0]:
silver_path = "abfss://retail@ecommerceadlsaccount.dfs.core.windows.net/silver/"

df_silver.write.mode("overwrite").format("delta").save(silver_path)

## create silver dataset

In [0]:
df_silver.write.mode("overwrite").format("delta").saveAsTable("retail_silver_cleaned")

In [0]:
%sql
select * from retail_silver_cleaned

store_id,product_id,customer_id,transaction_id,quantity,transaction_date,email,city,registration_date,full_name,product_name,category,price,store_name,location,total_amount
5,7,101,28,3,2024-11-15,user101@example.com,Delhi,2023-09-14,Ravi Yadav,Smartwatch,Electronics,4999.0,Mega Plaza,Chennai,14997.0
3,1,102,11,2,2024-08-11,user102@example.com,Mumbai,2024-01-21,Nina Joshi,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,1599.98
4,1,103,18,3,2024-09-05,user103@example.com,Bangalore,2023-07-10,Sonal Sharma,Wireless Mouse,Electronics,799.99,Downtown Mini Store,Pune,2399.97
3,3,104,13,4,2025-05-04,user104@example.com,Hyderabad,2024-02-05,Karan Patel,Yoga Mat,Fitness,499.0,Tech World Outlet,Bangalore,1996.0
3,1,105,21,5,2024-10-02,user105@example.com,Chennai,2023-06-28,Riya Singh,Wireless Mouse,Electronics,799.99,Tech World Outlet,Bangalore,3999.95
2,5,105,5,1,2025-03-17,user105@example.com,Chennai,2023-06-28,Riya Singh,Notebook Set,Stationery,149.0,High Street Store,Delhi,149.0
4,3,105,2,5,2024-11-12,user105@example.com,Chennai,2023-06-28,Riya Singh,Yoga Mat,Fitness,499.0,Downtown Mini Store,Pune,2495.0
3,9,107,22,4,2024-11-16,user107@example.com,Ahmedabad,2023-05-12,Priya Kapoor,Dumbbell Set,Fitness,1999.0,Tech World Outlet,Bangalore,7996.0
1,5,108,12,4,2025-05-26,user108@example.com,Kolkata,2023-08-19,Rahul Verma,Notebook Set,Stationery,149.0,City Mall Store,Mumbai,596.0
5,8,109,17,5,2024-07-10,user109@example.com,Delhi,2024-04-01,Pooja Mehta,Desk Organizer,Accessories,399.0,Mega Plaza,Chennai,1995.0
